# 03 - Video Inference: Detector -> Crop -> Classify -> Annotate

Video is the project's **actual use case**. This notebook builds the baseline video
pipeline with the classifier trained in `01_training.ipynb`:

```
input video
   |  (frame by frame)
   v
vehicle detection      pretrained detector (no detector training in this phase)
   v
vehicle crop           bounding box -> crop with a small margin
   v
vehicle classification Stanford Cars classifier from a checkpoint
                       (baseline from 01, or the distilled student from 04)
   v
draw results           box + predicted class + confidence
   v
output video
```

The **detector and classifier stay conceptually separate** - each is a small object with a
single method (`detect(frame)` / `predict(crops)`). That separation is what later lets us
swap in the distilled student, ONNX, and TensorRT versions without touching the pipeline.

Scope: plain frame-by-frame processing, no tracking, no optimization - that comes in later
phases. Requirements: a trained `models/baseline/best.pt` and any traffic video.

In [ ]:
import os, sys, time, textwrap

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision
import cv2
from PIL import Image

%matplotlib inline

print("Python        :", sys.version.split()[0])
print("PyTorch       :", torch.__version__)
print("torchvision   :", torchvision.__version__)
print("OpenCV        :", cv2.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU           :", torch.cuda.get_device_name(0))

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device        :", DEVICE)

## 2. Configuration

`INPUT_VIDEO` / `OUTPUT_VIDEO` can be overridden with environment variables of the same
names. Any video containing vehicles works - a short dashcam or street clip is enough.
(Processing a webcam instead is a one-line change: `cv2.VideoCapture(0)`.)

In [ ]:
from pathlib import Path

INPUT_VIDEO  = Path(os.environ.get("INPUT_VIDEO", "../data/videos/input.mp4"))
OUTPUT_VIDEO = Path(os.environ.get("OUTPUT_VIDEO", "../outputs/annotated.mp4"))

CHECKPOINT_PATH = Path(os.environ.get("CHECKPOINT_PATH", "../models/baseline/best.pt"))

DETECTOR_CONF_THRESHOLD = 0.5   # detector confidence threshold
CROP_MARGIN             = 0.10  # extra margin around each box, as a fraction of box size
MAX_PREVIEW_FRAMES      = 6     # annotated frames kept for the inline preview
PREVIEW_EVERY_N_FRAMES  = 50

print("INPUT_VIDEO     :", INPUT_VIDEO)
print("OUTPUT_VIDEO    :", OUTPUT_VIDEO)
print("CHECKPOINT_PATH :", CHECKPOINT_PATH)

## 3. Vehicle classifier (from a project checkpoint)

Loaded from the checkpoint - including the **preprocessing**, which is rebuilt from the
checkpoint dictionary so crops are normalized exactly like during training (the reuse
promised in notebook 01, section 4).

The default is the baseline from 01. To run the pipeline with the **distilled student from
04**, point `CHECKPOINT_PATH` at `models/distilled/best.pt` - the architecture is rebuilt
from what the checkpoint records, so no code changes are needed.

In [ ]:
from torchvision import transforms

MEAN = [0.485, 0.456, 0.406]   # ImageNet statistics
STD  = [0.229, 0.224, 0.225]

PREPROCESSING = {
    "image_size": IMAGE_SIZE,
    "mean": MEAN,
    "std": STD,
    "interpolation": "bilinear",
}

def build_transforms(preprocessing, augment):
    """Build the transform for one mode from a PREPROCESSING dict.

    augment=True  -> training: random crop + horizontal flip
    augment=False -> eval / inference: deterministic resize
    """
    size = preprocessing["image_size"]
    if augment:
        spatial = [transforms.RandomResizedCrop(size, scale=(0.6, 1.0)),
                   transforms.RandomHorizontalFlip()]
    else:
        spatial = [transforms.Resize((size, size))]
    return transforms.Compose(
        spatial
        + [transforms.ToTensor(),
           transforms.Normalize(preprocessing["mean"], preprocessing["std"])])

train_tf = build_transforms(PREPROCESSING, augment=True)
eval_tf  = build_transforms(PREPROCESSING, augment=False)
print("train transform:", train_tf)
print()
print("eval transform :", eval_tf)

In [ ]:
ckpt = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)  # our own checkpoint


def build_model_from_checkpoint(ckpt):
    """Rebuild any architecture recorded in a project checkpoint (baseline or student)."""
    arch = ckpt["model_arch"]
    if arch == "efficientnet_b0":
        from torchvision.models import efficientnet_b0
        model = efficientnet_b0(weights=None)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, ckpt["num_classes"])
    elif arch == "mobilenet_v3_small":
        from torchvision.models import mobilenet_v3_small
        model = mobilenet_v3_small(weights=None)
        model.classifier[3] = nn.Linear(model.classifier[3].in_features, ckpt["num_classes"])
    else:
        raise ValueError("unknown architecture: " + str(arch))
    return model


model = build_model_from_checkpoint(ckpt)
model.load_state_dict(ckpt["state_dict"])
model.to(DEVICE).eval()

print("classifier    :", ckpt["model_arch"], "-", ckpt["num_classes"], "classes")
print("preprocessing :", ckpt["preprocessing"])
print("val metrics   :", ckpt["metrics"])


class VehicleClassifier:
    """Stanford Cars classifier over BGR crops (as produced by OpenCV)."""

    def __init__(self, model, checkpoint, device):
        self.model = model
        self.classes = checkpoint["classes"]
        self.transform = build_transforms(checkpoint["preprocessing"], augment=False)
        self.device = device

    @torch.no_grad()
    def predict(self, crops_bgr):
        """crops_bgr: list of HxWx3 uint8 BGR arrays.
        Returns (class names, confidences) aligned with the input list."""
        if not crops_bgr:
            return [], []
        batch = torch.stack([
            self.transform(Image.fromarray(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)))
            for crop in crops_bgr
        ]).to(self.device)
        probs = torch.softmax(self.model(batch), dim=1)
        conf, pred = probs.max(dim=1)
        names = [self.classes[i] for i in pred.tolist()]
        return names, conf.cpu().tolist()


classifier = VehicleClassifier(model, ckpt, DEVICE)

## 4. Vehicle detector (pretrained, frozen)

The detector is an **existing pretrained model** - Faster R-CNN with a ResNet-50 FPN
backbone, trained on COCO (`torchvision` downloads the weights on first use). We only keep
the vehicle categories COCO knows:

- `car`, `bus`, `truck` - the vehicle types Stanford Cars covers (passenger vehicles).
  Add `"motorcycle"` to `VEHICLE_NAMES` if your videos contain bikes.

Training a custom detector is explicitly out of scope for this phase; because the detector
is isolated behind one method, it can be swapped later (e.g. for a TensorRT engine)
without touching anything else.

In [ ]:
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights

detector_weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
COCO_CATEGORIES = detector_weights.meta["categories"]

VEHICLE_NAMES = ["car", "bus", "truck"]
VEHICLE_IDS = {name: COCO_CATEGORIES.index(name) for name in VEHICLE_NAMES}


class VehicleDetector:
    """Pretrained COCO detector.

    detect(frame_bgr) -> list of (box xyxy in pixels, score, coco class name)
    """

    def __init__(self, weights, vehicle_ids, score_threshold, device):
        self.model = fasterrcnn_resnet50_fpn(weights=weights).to(device).eval()
        self.categories = weights.meta["categories"]
        self.vehicle_label_ids = set(vehicle_ids.values())
        self.score_threshold = score_threshold
        self.device = device

    @torch.no_grad()
    def detect(self, frame_bgr):
        rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        tensor = torch.from_numpy(rgb).permute(2, 0, 1).float() / 255.0
        output = self.model([tensor.to(self.device)])[0]
        results = []
        for box, label, score in zip(output["boxes"], output["labels"], output["scores"]):
            if int(label) in self.vehicle_label_ids and float(score) >= self.score_threshold:
                results.append((box.cpu().numpy(), float(score), self.categories[int(label)]))
        return results


detector = VehicleDetector(detector_weights, VEHICLE_IDS, DETECTOR_CONF_THRESHOLD, DEVICE)
print("detector          : fasterrcnn_resnet50_fpn (COCO)")
print("vehicle categories:", VEHICLE_IDS)

## 5. Pipeline: crop, classify, annotate

Connecting the two components per frame:

```
frame
 -> detector.detect(frame)          bounding boxes (car / bus / truck)
 -> crop_with_margin(frame, box)    vehicle crop
 -> classifier.predict(crops)       Stanford Cars class + confidence
 -> draw_detections(frame, ...)     annotated frame
```

In [ ]:
def crop_with_margin(frame, box, margin=CROP_MARGIN):
    """Crop the box region, expanded by `margin` on each side (clamped to the frame)."""
    h, w = frame.shape[:2]
    x1, y1, x2, y2 = box
    dx, dy = (x2 - x1) * margin, (y2 - y1) * margin
    x1, y1 = max(0, int(x1 - dx)), max(0, int(y1 - dy))
    x2, y2 = min(w, int(x2 + dx)), min(h, int(y2 + dy))
    return frame[y1:y2, x1:x2]


def analyze_frame(frame):
    """Full per-frame pipeline. Returns [(box, class name, confidence), ...]."""
    detections = detector.detect(frame)
    crops = [crop_with_margin(frame, box) for box, _, _ in detections]
    names, confs = classifier.predict(crops)
    return [(box, name, conf) for (box, _, _), name, conf in zip(detections, names, confs)]


def draw_detections(frame, detections):
    """Draw boxes + class + confidence. Modifies and returns the frame."""
    for box, name, conf in detections:
        x1, y1, x2, y2 = (int(v) for v in box)
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 200, 0), 2)
        text = textwrap.shorten(name, 32) + " %.0f%%" % (conf * 100)
        (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
        cv2.rectangle(frame, (x1, max(0, y1 - th - 8)), (x1 + tw + 6, y1), (0, 200, 0), -1)
        cv2.putText(frame, text, (x1 + 3, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5,
                    (0, 0, 0), 1, cv2.LINE_AA)
    return frame


print("pipeline ready")

## 6. Process the video

Frame-by-frame: read -> analyze -> annotate -> write. Progress is printed every 50 frames.
Set `INPUT_VIDEO` above (or the environment variable) to a real video before running.

In [ ]:
if not INPUT_VIDEO.exists():
    raise FileNotFoundError(
        "INPUT_VIDEO not found: " + str(INPUT_VIDEO)
        + "\nPlace a traffic video there or set the INPUT_VIDEO environment variable.")

capture = cv2.VideoCapture(str(INPUT_VIDEO))
if not capture.isOpened():
    raise RuntimeError("could not open video: " + str(INPUT_VIDEO))

fps   = capture.get(cv2.CAP_PROP_FPS) or 30.0
width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
total = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
print("input : %s (%dx%d @ %.1f fps, %d frames)" % (INPUT_VIDEO, width, height, fps, total))

OUTPUT_VIDEO.parent.mkdir(parents=True, exist_ok=True)
writer = cv2.VideoWriter(str(OUTPUT_VIDEO), cv2.VideoWriter_fourcc(*"mp4v"), fps, (width, height))
if not writer.isOpened():
    raise RuntimeError("could not open VideoWriter for " + str(OUTPUT_VIDEO)
                       + " (if the mp4v codec is unavailable, try XVID with an .avi output)")

preview_frames = []
frame_idx = 0
t0 = time.time()

while True:
    ok, frame = capture.read()
    if not ok:
        break
    detections = analyze_frame(frame)
    annotated = draw_detections(frame.copy(), detections)
    writer.write(annotated)

    if frame_idx % PREVIEW_EVERY_N_FRAMES == 0 and len(preview_frames) < MAX_PREVIEW_FRAMES:
        preview_frames.append(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))

    frame_idx += 1
    if frame_idx % 50 == 0:
        print("  frame %5d/%d" % (frame_idx, total))

capture.release()
writer.release()

elapsed = time.time() - t0
print()
print("processed %d frames in %.1fs (%.1f fps processing, video is %.1f fps)" % (frame_idx, elapsed, frame_idx / elapsed, fps))
print("output    :", OUTPUT_VIDEO, "(%.1f MB)" % (OUTPUT_VIDEO.stat().st_size / 1e6))

## 7. Preview

A few annotated frames sampled during processing (full result is the output video file).

In [ ]:
if preview_frames:
    fig, axes = plt.subplots(2, 3, figsize=(17, 7))
    for ax, img in zip(axes.ravel(), preview_frames):
        ax.imshow(img)
        ax.axis("off")
    for ax in list(axes.ravel())[len(preview_frames):]:
        ax.axis("off")
    plt.suptitle("Sampled annotated frames", y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("no preview frames captured")

## 8. Summary and next steps

This is the **baseline video pipeline**: a frozen pretrained detector, the 01 classifier,
frame-by-frame processing, no tracking and no optimization - by design.

Later phases build directly on this structure:

| phase | change |
|---|---|
| Knowledge Distillation | load `models/distilled/best.pt` via `CHECKPOINT_PATH` - already works |
| ONNX + TensorRT | `detect` / `predict` swap their backend engines; pipeline code unchanged |
| Tracking | add identity assignment between frames on top of the boxes |
| Jetson Orin Nano | same pipeline, benchmarked for real-time FPS |